# Neuronpedia Feature Explanation Lookup Table

**Dataset**: 6,574 unique CLT features extracted from 8 representative attribution graphs (one per capability domain) via Cantor decode, with semantic explanations fetched from Neuronpedia API.

**What this notebook does**:
1. Loads the pre-built feature explanation dataset (mini subset of 45 diverse examples)
2. Parses feature records containing explanations, activation statistics, and graph provenance
3. Analyzes feature distribution across layers and capability domains
4. Visualizes activation patterns and domain coverage

**Model**: gemma-2-2b | **Source**: Neuronpedia API | **Domains**: antonym, arithmetic, code_completion, country_capital, multi_hop_reasoning, rhyme, sentiment, translation

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import json
import os
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate

## Load Data

Load the mini demo dataset from GitHub (with local fallback for development).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-582cc7-circuit-motif-spectroscopy-discovering-u/main/dataset_iter2_neuronpedia_fea/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded dataset: {data['metadata']['description']}")
print(f"Model: {data['metadata']['model']}")
print(f"Features in this subset: {len(data['datasets'][0]['examples'])}")
print(f"Source domains: {', '.join(data['metadata']['source_domains'])}")

## Configuration

Tunable parameters for this demo.

In [ ]:
# --- Config ---
MAX_EXAMPLES = 45       # Max number of examples to process (full dataset: 6574)
TOP_N_DISPLAY = 10      # Number of top features to show in summary tables
TOP_N_LOGITS = 5        # Number of top logits to display per feature

## Parse Feature Records

Each example's `output` field is a JSON string containing the full feature details. We parse these into structured records.

In [ ]:
examples = data['datasets'][0]['examples'][:MAX_EXAMPLES]

# Parse the output JSON strings into dicts
records = []
for ex in examples:
    output = json.loads(ex['output'])
    output['input_key'] = ex['input']
    output['metadata_source_domains'] = ex['metadata_source_domains']
    records.append(output)

print(f"Parsed {len(records)} feature records")
print(f"Sample record keys: {list(records[0].keys())}")

## Build Summary Statistics

Count features per layer and per capability domain, and compute explanation coverage.

In [ ]:
# Count features per layer
layer_counts = Counter(r['layer_num'] for r in records)

# Count features per domain
domain_counts = Counter()
for r in records:
    for d in r['metadata_source_domains'].split(','):
        if d:
            domain_counts[d] += 1

# Explanation coverage
n_with_expl = sum(1 for r in records if r['has_explanation'])

print(f"Total features: {len(records)}")
print(f"With explanation: {n_with_expl} ({n_with_expl/len(records)*100:.1f}%)")
print(f"Unique layers: {len(layer_counts)}")
print(f"Unique domains: {len(domain_counts)}")
print()

# Layer distribution table
print("Features per layer:")
layer_table = [[layer, count] for layer, count in sorted(layer_counts.items())]
print(tabulate(layer_table, headers=["Layer", "Count"], tablefmt="simple"))
print()

# Domain distribution table
print("Features per domain:")
domain_table = [[d, c] for d, c in sorted(domain_counts.items(), key=lambda x: -x[1])]
print(tabulate(domain_table, headers=["Domain", "Count"], tablefmt="simple"))

## Top Feature Explanations

Display the most activated features with their semantic explanations and source domains.

In [ ]:
# Sort by max_activation descending
sorted_records = sorted(records, key=lambda r: r.get('max_activation') or 0, reverse=True)

top_table = []
for r in sorted_records[:TOP_N_DISPLAY]:
    top_logits = r.get('top_positive_logits', [])[:TOP_N_LOGITS]
    top_table.append([
        r['input_key'],
        f"L{r['layer_num']}",
        r.get('explanation', '')[:60],
        f"{r.get('max_activation', 0):.1f}",
        f"{r.get('frac_nonzero', 0):.4f}",
        ', '.join(r.get('source_domains', [])),
        ', '.join(top_logits[:3]),
    ])

print(f"Top {TOP_N_DISPLAY} features by max activation:")
print(tabulate(top_table,
    headers=["Key", "Layer", "Explanation", "MaxAct", "FracNZ", "Domains", "TopLogits"],
    tablefmt="simple", maxcolwidths=[None, None, 60, None, None, 20, 25]))

## Visualization

Visualize feature distribution across layers, domains, and activation statistics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Features per layer
layers_sorted = sorted(layer_counts.keys())
ax = axes[0, 0]
ax.bar(layers_sorted, [layer_counts[l] for l in layers_sorted], color='steelblue', edgecolor='white')
ax.set_xlabel('Layer')
ax.set_ylabel('Feature Count')
ax.set_title('Features per Layer')

# 2. Features per domain
domains_sorted = sorted(domain_counts.keys(), key=lambda d: -domain_counts[d])
ax = axes[0, 1]
ax.barh(domains_sorted, [domain_counts[d] for d in domains_sorted], color='coral', edgecolor='white')
ax.set_xlabel('Feature Count')
ax.set_title('Features per Capability Domain')

# 3. Max activation distribution
max_acts = [r.get('max_activation') for r in records if r.get('max_activation') is not None]
ax = axes[1, 0]
ax.hist(max_acts, bins=20, color='seagreen', edgecolor='white', alpha=0.8)
ax.set_xlabel('Max Activation')
ax.set_ylabel('Count')
ax.set_title('Distribution of Max Activation')

# 4. Fraction nonzero vs max activation (scatter)
frac_nz = [r.get('frac_nonzero') for r in records if r.get('frac_nonzero') is not None and r.get('max_activation') is not None]
max_act_scatter = [r.get('max_activation') for r in records if r.get('frac_nonzero') is not None and r.get('max_activation') is not None]
n_domains = [len(r.get('source_domains', [])) for r in records if r.get('frac_nonzero') is not None and r.get('max_activation') is not None]
ax = axes[1, 1]
scatter = ax.scatter(frac_nz, max_act_scatter, c=n_domains, cmap='viridis', alpha=0.7, edgecolors='gray', s=60)
ax.set_xlabel('Fraction Nonzero')
ax.set_ylabel('Max Activation')
ax.set_title('Activation vs Sparsity (color = # domains)')
plt.colorbar(scatter, ax=ax, label='# Source Domains')

plt.tight_layout()
plt.savefig('feature_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved visualization to feature_analysis.png")